In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
from openai import OpenAI
openai_client = OpenAI()

In [6]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [7]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Maybe — it depends on the course’s enrollment rules and how far along it is.

If you mean:
- **Live course with a schedule:** you may be able to join if registration is still open, or if the instructor allows late enrollment.
- **Self-paced course:** you can usually join anytime.
- **School/class-based course:** you may need approval from the instructor or admin office.

If you want, I can help you draft a quick message like:

> Hi, I just discovered the course. Is it still possible to join at this point? I’d love to enroll if there’s still availability.

If you tell me the course type, I can give a more specific answer.


In [8]:
# Add context to answer the questions students might have
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [9]:
# Configure the main function of the LLM model
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}

"""

In [10]:
print(prompt)


Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
I just discovered the course. Can I join now?

Context:

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students partici

In [11]:
question = "I just discovered the course. Can I join now?"
answer = llm(prompt)
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.


In [12]:
# We would like to implement RAG, which includes 3 steps:
def rag(question):
    # Retrieval or Search
    search_results = search(question)
    # Augmented
    user_prompt = build_prompt(question, search_results)
    # Generation (sending the result to an LLM)
    return llm(user_prompt)

## Step 1 = Define Search

In [13]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [14]:
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [15]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    # raise_for_status - if something is broken, raise an error 
    # and do not continue
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1342

In [16]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [17]:
# This looks easy, but realistically speaking, we will need to invest
# a good amount of effort into data pre-processing (here it was in the shape of a .json) to obtain a single document
# in this case, for llm course, we will use this link: https://datatalks.club/faq/json/llm-zoomcamp.json

In [18]:
# For the next step, we need to use a library that will allow us to perform a search in the knowledge database, index it
# so that we don't send all of the documents to the LLM, 
# but only end the most relevant chunks for the LLM to produce the final result


In [19]:
### Let's look for top candidates first! - lucene, elasticsearch, solr. We will use minsearch
from minsearch import Index


index = Index(
    # text fields are all fields that can be used to perform text
    #  search ('section', 'question', 'answer' from documents[0])
    text_fields=["question", "section", "answer"],
    # similar to a WHERE clause in a query like this:
    # SELECT * FROM index WHERE course = 'data-engineering-zoomcamp
    keyword_fields=["course"]
)

index.fit(documents)


In [20]:
index.search(
    question, 
    filter_dict={'course':'llm-zoomcamp'}, 
    num_results=3
    )

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [21]:
# let's define the search step:
def search(question, course = 'llm-zoomcamp'):

     # basically, if we see a word 'certificate' in the question,
    # it is twice as important that only seeing it in the answer section
    boost_dict = {'question':2.0, 'section': 0.5}
    filter_dict = {'course':course}

    return index.search(
    question, 
    boost_dict=boost_dict, 
    filter_dict=filter_dict, 
    num_results=3
    )

In [22]:
question = "I just discovered the course. Can I join now?"
search_results = search(question)

In [23]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

## Step 2 = Building the Prompt

In [24]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}

"""

Building a prompt consists of 2 parts: instructions (do not change) and user prompt (changes with every request)

In [25]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."""

In [26]:
USER_PROMPT_TEMPLATE= f"""

Question:
{question}

Context:
{context}
"""

In [27]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [28]:
context = build_context(search_results)
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is run

In [29]:
def build_prompt(question, search_results):
    context = build_context(search_results = search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question = question, 
        context=context
        )
    return prompt.strip()

In [30]:
question = "I just discovered the course. Can I join now?"
prompt = build_prompt(question, search_results)

In [31]:
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.


## Step 3: LLM a.k.a Generation part of RAG

In [ ]:
# Let's examine the output body and decompose response into 2 parts
response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )

In [33]:
# shortcut way to get the human-readable output
response.output_text

'Yes — you can still join now.\n\nIf you want to receive a certificate, make sure to submit your project while submissions are still open.'

In [34]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_0a80e153f080a241006a30353e9288819eb11c42624773e75c",
  "created_at": 1781544254.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.4-mini-2026-03-17",
  "object": "response",
  "output": [
    {
      "id": "msg_0a80e153f080a241006a30353fdaac819eab64f7ee13be9674",
      "content": [
        {
          "annotations": [],
          "text": "Yes — you can still join now.\n\nIf you want to receive a certificate, make sure to submit your project while submissions are still open.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": "final_answer"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 0.98,
  "background": false,
  "completed_at": 1781544256.0,
  "conversation": null,
  "max_output_tokens": null,
  "max_tool_calls": null,


In [35]:
# tells us the number of input, caches, and output tokens used for this request
response.usage

ResponseUsage(input_tokens=200, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=32, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=232)

In [36]:
# can figure out how much to pay for this request by going to 
# here: https://developers.openai.com/api/docs/models/gpt-5.4-mini for the mini model
# and seeing the price breakdown: $0.75 for input, $0.075 for cached, and $4.5 for output per million

input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000
cached_price = 0.075 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price +
    response.usage.input_tokens_details.cached_tokens * cached_price
)

cost

0.000294

In [37]:
# For response, we take the prompt, which is just a string
# and send it to OpenAI
# There are 2 API's OpenAI has: chat completions and responses API
# Responses is a newer API


# The response API typically has access to the conversation history (not stored unless you have memeory enabled)

message_history = [
    # system prompt where INSTRUCTIONS are immutable
    {'role': 'developer', 'content': INSTRUCTIONS},
    # user prompt where prompt is MUTABLE with each request
    {'role': 'user', 'content': prompt},
]

# place message history into response

response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=message_history
    )

In [38]:
response.output_text

'Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.'

In [39]:
def llm(instructions, user_prompt, model = "gpt-5.4-mini"):
    message_history = [
        # system prompt where INSTRUCTIONS are immutable
        {'role': 'developer', 'content': instructions},
        # user prompt where prompt is MUTABLE with each request
        {'role': 'user', 'content': user_prompt},
    ]

# place message history into response

    response = openai_client.responses.create(
            model=model,
            input=message_history
        ) 
    return response.output_text

In [40]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model = model)
    return answer

In [41]:
answer = rag(question)
print(answer)

Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.
